In [2]:
import numpy as np
import pandas as pd

In [4]:
weights = np.array([
    [-1.8, -0.9, 0.0, 0.7, 1.5],
    [-2.4, -0.3, 0.2, 1.1, 2.0]
], dtype=np.float32)

activations = np.array([
    [0.0, 0.3, 0.8, 1.4, 2.1],
    [0.1, 0.6, 1.0, 1.8, 3.2]
], dtype=np.float32)

In [32]:
def symmetric_quantize(tensor):
    max_abs = np.max(np.abs(tensor))

    if max_abs == 0:
        scale = 1.0
    else:
        scale = max_abs / 127

    zero_point = 0

    q = np.round(tensor / scale)

    q = np.clip(q, -127, 127)
    

    return q.astype(np.int8), scale, zero_point

In [8]:
def asymmetric_quantize(tensor):

    q_min = -128
    q_max = 127

    x_min = np.min(tensor)
    x_max = np.max(tensor)

    if np.isclose(x_min, x_max):
        scale = 1.0
        zero_point = 0
    else:
        scale = (x_max - x_min) / (q_max - q_min)

        zero_point = np.round(q_min - (x_min / scale))
        zero_point = np.clip(zero_point, q_min, q_max)

    q = np.round(tensor / scale) + zero_point

    q = np.clip(q, q_min, q_max)

    return q.astype(np.int8), scale, int(zero_point)

In [10]:
def dequantize(q_tensor, scale, zero_point):

    return (q_tensor.astype(np.float32) - zero_point) * scale

In [12]:
def calculate_metrics(original, reconstructed, quantized):

    abs_error = np.abs(original - reconstructed)

    mae = np.mean(abs_error)

    mse = np.mean((original - reconstructed) ** 2)

    max_error = np.max(abs_error)

    sat_min = np.sum(quantized == -128)

    sat_max = np.sum(quantized == 127)

    sat_total = sat_min + sat_max

    return mae, mse, max_error, sat_min, sat_max, sat_total

In [14]:
def compare_tensor(name, tensor):

    rows = []

    for method in ["Symmetric", "Asymmetric"]:

        if method == "Symmetric":
            q, scale, zp = symmetric_quantize(tensor)
        else:
            q, scale, zp = asymmetric_quantize(tensor)

        dq = dequantize(q, scale, zp)

        mae, mse, maxerr, satmin, satmax, sattotal = calculate_metrics(
            tensor,
            dq,
            q
        )

        rows.append([
            name,
            method,
            scale,
            zp,
            mae,
            mse,
            maxerr,
            satmin,
            satmax,
            sattotal
        ])

    return pd.DataFrame(
        rows,
        columns=[
            "Tensor",
            "Method",
            "Scale",
            "Zero Pt",
            "MAE",
            "MSE",
            "Max Err",
            "Sat(min)",
            "Sat(max)",
            "Sat(total)"
        ]
    )

In [16]:
weights_table = compare_tensor("Weights", weights)

activations_table = compare_tensor("Activations", activations)

print(weights_table)

print()

print(activations_table)

    Tensor      Method     Scale  Zero Pt       MAE       MSE   Max Err  \
0  Weights   Symmetric  0.018898        0  0.003701  0.000022  0.007874   
1  Weights  Asymmetric  0.017255       11  0.003804  0.000021  0.007451   

   Sat(min)  Sat(max)  Sat(total)  
0         0         0           0  
1         1         1           2  

        Tensor      Method     Scale  Zero Pt       MAE       MSE   Max Err  \
0  Activations   Symmetric  0.025197        0  0.005276  0.000045  0.011024   
1  Activations  Asymmetric  0.012549     -128  0.002627  0.000011  0.005490   

   Sat(min)  Sat(max)  Sat(total)  
0         0         1           1  
1         1         1           2  


In [18]:
def display_results(name, tensor):

    sym_q, sym_scale, sym_zp = symmetric_quantize(tensor)

    sym_dq = dequantize(sym_q, sym_scale, sym_zp)

    asym_q, asym_scale, asym_zp = asymmetric_quantize(tensor)

    asym_dq = dequantize(asym_q, asym_scale, asym_zp)

    print("="*60)
    print(name)
    print("="*60)

    print("\nOriginal")
    print(tensor)

    print("\nSymmetric Quantized")
    print(sym_q)

    print("\nSymmetric Dequantized")
    print(sym_dq)

    print("\nAsymmetric Quantized")
    print(asym_q)

    print("\nAsymmetric Dequantized")
    print(asym_dq)

In [20]:
display_results("Weights", weights)

display_results("Activations", activations)

Weights

Original
[[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]

Symmetric Quantized
[[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]

Symmetric Dequantized
[[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]

Asymmetric Quantized
[[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]

Asymmetric Dequantized
[[-1.7945098  -0.8972549   0.          0.707451    1.5011765 ]
 [-2.3984313  -0.29333332  0.20705882  1.1043137   2.0015686 ]]
Activations

Original
[[0.  0.3 0.8 1.4 2.1]
 [0.1 0.6 1.  1.8 3.2]]

Symmetric Quantized
[[  0  12  32  56  83]
 [  4  24  40  71 127]]

Symmetric Dequantized
[[0.        0.3023622 0.8062992 1.4110236 2.0913386]
 [0.1007874 0.6047244 1.007874  1.7889764 3.2      ]]

Asymmetric Quantized
[[-128 -104  -64  -16   39]
 [-120  -80  -48   15  127]]

Asymmetric Dequantized
[[0.         0.30117646 0.80313724 1.4054902  2.0956862 ]
 [0.10039216 0.6023529  1.0039215  1.79

In [22]:
outlier_tensor = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7, 12.0],
    dtype=np.float32
)

without_outlier = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7],
    dtype=np.float32
)

In [24]:
print(compare_tensor("With Outlier", outlier_tensor))

print()

print(compare_tensor("Without Outlier", without_outlier))

         Tensor      Method     Scale  Zero Pt       MAE       MSE   Max Err  \
0  With Outlier   Symmetric  0.094488        0  0.015617  0.000441  0.038583   
1  With Outlier  Asymmetric  0.049020     -118  0.007190  0.000072  0.013725   

   Sat(min)  Sat(max)  Sat(total)  
0         0         1           1  
1         1         1           2  

            Tensor      Method     Scale  Zero Pt       MAE       MSE  \
0  Without Outlier   Symmetric  0.005512        0  0.001102  0.000002   
1  Without Outlier  Asymmetric  0.004706      -22  0.001176  0.000002   

    Max Err  Sat(min)  Sat(max)  Sat(total)  
0  0.002362         0         1           1  
1  0.002353         1         1           2  


# Observations

## Symmetric Quantization

- Symmetric quantization uses a **zero point of 0** and calculates the scale using the maximum absolute value of the tensor.
- For the **weights** tensor, the reconstructed values are very close to the original values, with only small differences caused by rounding.
- For the **activations** tensor, symmetric quantization also produces accurate results, but because the tensor contains only positive values, part of the negative INT8 range remains unused.
- Symmetric quantization is simple to implement and is commonly used for quantizing model weights.

---

## Asymmetric Quantization

- Asymmetric quantization calculates both the **scale** and **zero point** from the minimum and maximum values of the tensor.
- This allows the full INT8 range to be utilized even when the data is not centered around zero.
- For the **weights** tensor, asymmetric quantization produced reconstruction quality very similar to symmetric quantization.
- For the **activations** tensor, asymmetric quantization used the available quantization range more effectively by shifting the zero point to **-128**, resulting in slightly more accurate reconstructed values.

---

## Weights Comparison

- The weights contain both positive and negative values, making them naturally suited for symmetric quantization.
- Both methods reconstructed the weights with very small errors.
- The dequantized tensors from both methods closely matched the original tensor, indicating that either method can represent the weights accurately.
- Since symmetric quantization is simpler and provides comparable accuracy, it is generally preferred for weight tensors.

---

## Activations Comparison

- The activations contain only non-negative values.
- Symmetric quantization achieved accurate reconstruction, but half of the available INT8 range was effectively unused.
- Asymmetric quantization shifted the zero point, allowing the entire INT8 range to represent positive activation values more efficiently.
- The reconstructed activations from both methods were close to the original values, with asymmetric quantization providing slightly better utilization of the quantization range.

---

## Outlier Experiment

### With Outlier (12.0)

- The large outlier increased the quantization scale significantly.
- **Symmetric quantization**
  - Scale = **0.094488**
  - MAE = **0.015617**
  - MSE = **0.000441**
  - Maximum Error = **0.038583**
- **Asymmetric quantization**
  - Scale = **0.049020**
  - MAE = **0.007190**
  - MSE = **0.000072**
  - Maximum Error = **0.013725**
- In this case, asymmetric quantization produced noticeably lower reconstruction errors because it adapted the zero point to the tensor's range.

### Without Outlier

- Removing the outlier reduced the scale considerably.
- **Symmetric quantization**
  - Scale = **0.005512**
  - MAE = **0.001102**
  - MSE = **0.000002**
- **Asymmetric quantization**
  - Scale = **0.004706**
  - MAE = **0.001176**
  - MSE = **0.000002**
- After removing the outlier, both methods achieved very low reconstruction errors, and the difference between them became negligible.

---

## Saturation Analysis

- For the **weights** and **activations** tensors, only a few values reached the quantization limits, indicating that most values were represented without clipping.
- In the outlier experiment, saturation occurred mainly because the extreme value occupied the maximum representable INT8 level.
- The saturation counts confirm that outliers reduce the effective precision available for the remaining values.

---

## Conclusion

- **Symmetric quantization** is well suited for weight tensors because they are generally centered around zero and can be represented accurately with a zero point of 0.
- **Asymmetric quantization** is better suited for activation tensors because the adjustable zero point allows more efficient use of the available INT8 range.
- The outlier experiment clearly demonstrates that extreme values increase the quantization scale, reducing precision for smaller values and increasing reconstruction error.
- Removing the outlier significantly improved quantization accuracy for both methods, highlighting the impact of data distribution on quantization performance.ive precision for the majority of the data.
```